# Code 11.5.1: A reward model and its losses

`RewardModel` wraps a transformer backbone (`AutoModel`, which omits the language-modeling head) and reads a scalar at the last real token, assuming right padding. The head is initialized with standard deviation $`1/\sqrt{d+1}`$, following the reference implementation of Stiennon et al. (2020) as documented by Huang et al. (2024). `bt_loss` is the pairwise Bradley-Terry loss with an optional margin, and `ranking_loss` averages it over all pairs of a ranking.


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoModel

class RewardModel(nn.Module):
    """A transformer backbone with a scalar head read out at the last real token."""
    def __init__(self, name):
        super().__init__()
        self.backbone = AutoModel.from_pretrained(name)      # e.g. the SFT model without its LM head
        d = self.backbone.config.hidden_size
        self.head = nn.Linear(d, 1)
        nn.init.normal_(self.head.weight, std=1 / (d + 1) ** 0.5)
        nn.init.zeros_(self.head.bias)

    def forward(self, input_ids, attention_mask):
        h = self.backbone(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state
        last = attention_mask.sum(dim=1) - 1                 # index of the last real token (right padding)
        h_last = h[torch.arange(h.size(0)), last]
        return self.head(h_last).squeeze(-1)                 # one scalar per sequence

def bt_loss(r_chosen, r_rejected, margin=0.0):
    """Bradley-Terry pairwise loss: -log sigmoid(r_c - r_r - m)."""
    return -F.logsigmoid(r_chosen - r_rejected - margin).mean()

def ranking_loss(rewards):
    """InstructGPT-style loss for one prompt with K responses ranked best first:
    the pairwise loss averaged over all K*(K-1)/2 pairs."""
    K = rewards.shape[0]
    i, j = torch.triu_indices(K, K, offset=1)                # response i is ranked above response j
    return -F.logsigmoid(rewards[i] - rewards[j]).mean()

print(ranking_loss(torch.tensor([2.0, 1.0, 0.5, -1.0])))     # tensor(0.2276)
